instalacja dependencies

In [1]:
%pip install git+https://github.com/ibm-granite-community/utils \
  sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer

%pip install --no-deps bitsandbytes \
  accelerate \
  xformers==0.0.29.post3 \
  peft \
  trl \
  tqdm \
  triton \
  cut_cross_entropy \
  unsloth_zoo \
  unsloth

  Cloning https://github.com/ibm-granite-community/utils to /tmp/pip-req-build-14wfl0_0
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite-community/utils /tmp/pip-req-build-14wfl0_0
  Resolved https://github.com/ibm-granite-community/utils to commit aa05c43dc5ee022083221f3db59adc2ec869d50a
  Installing build dependencies ... canceled
ERROR: Operation cancelled by user
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 30.2 MB/s eta 0:00:00


ładowanie modelu bazowego


In [2]:
from unsloth import FastLanguageModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="ibm-granite/granite-3.3-8b-instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = False
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Granite patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.co

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

ibm-granite/granite-3.3-8b-instruct does not have a padding token! Will use pad_token = <|end_of_text|>.


grupowanie logów

In [1]:
import json
from datetime import datetime
from collections import defaultdict

with open("employee_activity_logs.json", "r") as f:
    raw_logs = json.load(f)

def parse_date(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00")).date().isoformat()

episodes = defaultdict(list)
for ev in sorted(raw_logs, key=lambda x: x["timestamp"]):
    emp = ev["employee_id"]
    sess = ev.get("session_id")
    if sess:
        key = f"{emp}__{sess}"
    else:
        key = f"{emp}__{parse_date(ev['timestamp'])}"
    episodes[key].append(ev)

len(episodes)

8

formatowanie logów dla promptu

In [4]:
def format_episode(key, events):
    emp, sess_or_date = key.split("__", 1)
    header = f"Employee: {emp}\nEpisode: {sess_or_date}\n\n"
    lines = []
    for ev in events:
        line = (
            f"{ev['timestamp']} | {ev['action_type']}"
            f" | resource={ev.get('resource_accessed')}"
            f" | type={ev.get('resource_type')}"
            f" | status={ev.get('request_status')}"
            f" | ip={ev.get('ip_address')}"
            f" | geo={ev.get('geo_location')}"
            f" | device={ev.get('device_fingerprint')}"
        )
        lines.append(line)
    return header + "\n".join(lines)

prompt główny

In [5]:
SYSTEM_PROMPT = (
    "You are a security and behavior analytics assistant. "
    "You analyze employee activity logs and extract high-level behavior patterns, "
    "focusing on what employees do, when, from where, and which resources they use. "
    "You ignore low-level technical metrics unless they are important for behavior."
)

tworzenie datasetu
1. target answers

In [6]:
TARGET_ANSWERS = {
    "sess_abc123": """The employee logged in from the corporate office network in Warsaw using a managed Windows desktop during normal business hours.
They accessed and read an internal Q4 financial report and then ended the session shortly afterwards.
There is no indication of abnormal access patterns, unusual devices, or off‑hours activity in this session.
Severity: normal.
Probable cause: regular business activity related to financial reporting and analysis.
Owner: Finance department in coordination with the employee's line manager.""",

    "sess_def456": """The employee logged in from the Krakow office using a corporate Mac laptop and successfully authenticated.
Shortly after logging in, they downloaded a data file named "employee_salaries_2025.csv", which likely contains highly sensitive salary information for staff.
The entire activity took place from a known corporate IP and device and the session was closed cleanly with a normal logout.
While the network location and device look legitimate, the type of data accessed represents a high‑sensitivity asset and may not be required for normal day‑to‑day duties.
Severity: suspicious.
Probable cause: intentional access to sensitive HR compensation data that may or may not be authorized for this user (potential data misuse or early stage data exfiltration).
Owner: Human Resources and Security/Compliance teams should review this access with the employee's manager.""",

    "sess_ghi789": """The account emp_003 logged in from an external IP address (85.232.45.100) with the user agent "curl/7.68.0", indicating scripted or command‑line access rather than a standard browser.
The geographic location is reported as "Unknown" and the device fingerprint is also unknown, which is inconsistent with typical managed corporate endpoints.
After logging in, the user downloaded a highly sensitive document named "company_confidential_strategy.pdf".
This combination of non‑interactive tooling, unrecognized device, unknown location, and access to a confidential strategic document is strongly indicative of a potential compromise or deliberate data theft.
Severity: critical.
Probable cause: stolen or misused credentials being used via an automated script from outside the corporate network, with the goal of exfiltrating confidential strategic information.
Owner: Security Incident Response Team (SIRT) and the CISO organization should immediately investigate, revoke active sessions, review other access by this account, and notify the document owners.""",

    "sess_jkl012": """The employee emp_001 logged in around 02:15 local time from Berlin, Germany using an iPhone mobile device, which suggests remote or travel‑related access outside normal office hours.
During this session, the employee read a document named "hr_layoff_plans.docx", which appears to be a highly sensitive HR planning file.
The connection was successful, the actions completed without errors, and the session ended with a normal logout a few minutes later.
Although the access is successful and from a plausible consumer device, the off‑hours timing, foreign city, and highly sensitive HR content make this session stand out from typical office activity.
Severity: suspicious.
Probable cause: legitimate employee remotely reviewing confidential HR plans while traveling or working off‑hours, but with a non‑negligible risk of unauthorized disclosure or someone else using the employee's device.
Owner: Human Resources and the employee's manager should confirm whether this access is aligned with the employee's role; Security should monitor for similar patterns from this device and location.""",

    "emp_004_sequence": """The account emp_004 initiated a login from the corporate address 10.10.10.10 using a Python‑based client ("Python-requests/2.28.0"), which is more typical of scripts or automated tools than of interactive user sessions.
Following an initial successful login, there were multiple consecutive failed_login events with HTTP 401 responses from the same IP and user agent, indicating repeated authentication failures.
Shortly afterwards, a password_change action succeeded under session sess_mno345 from the same IP and user agent.
This pattern — scripted access, repeated login failures, and then a successful password change — may indicate either an internal automation tool behaving unexpectedly or an attacker brute‑forcing or abusing credentials before resetting the password.
Severity: suspicious.
Probable cause: possible credential stuffing or scripted misuse of the emp_004 account, followed by a password reset that may or may not have been initiated by the legitimate owner.
Owner: Security Operations and Identity/Access Management teams should verify whether the password change was user‑initiated, review MFA logs, and potentially force a credential reset and device hygiene check for this user.""",

    "sess_pqr678": """The employee emp_005 logged in from the Warsaw office network using a managed Windows workstation with a known device fingerprint.
Soon after login, they downloaded a file named "source_code_backup.zip", which likely contains a backup archive of application source code or repositories.
All activity occurred during standard afternoon working hours from a recognized corporate IP and device with no failed attempts or anomalies recorded in this session.
Accessing source code backups can be a legitimate part of development, maintenance, or backup validation work, depending on the employee's role.
Severity: normal, assuming the employee works in engineering or a related technical role.
Probable cause: routine engineering or operations task involving retrieval of source code backups for development, debugging, or deployment purposes.
Owner: Engineering/DevOps leadership responsible for the associated application, together with the employee's direct manager.""",

    "sess_stu901": """Later the same day, the same employee emp_005 initiated a new session from an external IP address (176.123.89.45) using an Android device with an unknown device fingerprint, rather than the previously seen corporate Windows desktop.
From this unrecognized mobile device and unknown geographic location, the user logged in successfully and downloaded a highly sensitive data file named "customer_database_dump.sql", which likely contains a bulk export of customer records.
The combination of off‑premises access, untrusted mobile endpoint, and bulk database dump retrieval represents a strong data exfiltration risk.
The timing, device change, and nature of the file make it unlikely to be normal day‑to‑day activity, even if the credentials themselves are valid.
Severity: critical.
Probable cause: high‑risk data exfiltration attempt by either a malicious insider or an external attacker using the employee's credentials on an unmanaged mobile device.
Owner: Security Incident Response Team and Data Protection/Privacy Office should treat this as a potential breach, investigate scope and impact, and coordinate with the application owner responsible for the customer database."""
}

2. formulowanie datasetu

In [7]:
def get_target_analysis_for_episode(key, events):
    if any(ev.get("employee_id") == "emp_004" for ev in events):
        return TARGET_ANSWERS.get("emp_004_sequence")

    session_ids = {ev.get("session_id") for ev in events if ev.get("session_id")}
    for sess_id in session_ids:
        if sess_id in TARGET_ANSWERS:
            return TARGET_ANSWERS[sess_id]

    return None


training_examples = []

for key, events in episodes.items():
    target_analysis = get_target_analysis_for_episode(key, events)
    if target_analysis is None:
        continue

    logs_text = format_episode(key, events)

    example = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    "Here is one employee's activity episode.\n"
                    "1) Describe the employee's behavior at a high level.\n"
                    "2) List concrete behavior patterns.\n"
                    "3) Assess the risk level and explain why.\n\n"
                    f"{logs_text}"
                ),
            },
            {"role": "assistant", "content": target_analysis},
        ]
    }
    training_examples.append(example)

In [8]:
import json

with open("employee_behavior_train.jsonl", "w") as f:
    for ex in training_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

ładowanie datasetu


In [11]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer)


def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


In [12]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "employee_behavior_train.jsonl"},
    split="train",
)
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

LoRA fine tuning


In [13]:
target_modules =  ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

model = FastLanguageModel.get_peft_model(
    base_model,
    r = 16, # Rank of lora matrices
    target_modules = target_modules,  # Modules of the llm the lora weights are used
    lora_alpha = 16, # scales the weights of the adapters
    lora_dropout = 0, # Unsloth recommends 0 is better for fast patching
    bias = "none",    # "none" is optimized
    use_gradient_checkpointing = "unsloth", #"unsloth" for very long context, decreases vram
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients
